# 🔬 Cuaderno de Verificación Analítica: GWAS para Trastorno Bipolar (bpd)
**Módulo:** Prácticum  
**Tecnología Principal:** Polars (Engine escrito en Rust) + Plotly Interno  

Este cuaderno automatiza la auditoría de calidad y el filtrado masivo del dataset `bpd.parquet` (6.2 millones de registros). A diferencia de los enfoques tradicionales con Pandas, aquí implementamos **Evaluación Lazy (Perezosa)** para optimizar el uso de memoria RAM y acelerar las consultas mediante operaciones vectorizadas en paralelo.
---

In [1]:
# ── Dependencias ──────────────────────────────────────────────────────────────
# !pip install pandas pyarrow plotly seaborn matplotlib

In [2]:
import polars as pl
import os

# Definir la ruta relativa exacta del archivo de tu Prácticum
ruta_parquet = "../data/analysis/bpd.parquet"

print(f"¿El archivo existe?: {os.path.exists(ruta_parquet)}")
print(f"Tamaño del archivo: {os.path.getsize(ruta_parquet) / 1024**2:.2f} MB")

¿El archivo existe?: True
Tamaño del archivo: 281.10 MB


## 1. 📂 Celda 2: Comparación de Tiempo (Eager vs Lazy)

In [ ]:
import time

# Prueba 1: Carga Eager tradicional (Lee todo el archivo a RAM)
t0 = time.time()
df_eager = pl.read_parquet(ruta_parquet)
t_eager = time.time() - t0
print(f"⏱️ Tiempo de carga Completa en RAM (Eager): {t_eager:.4f} segundos")

# Prueba 2: Escaneo Lazy Inteligente (Instante, no consume RAM)
t0 = time.time()
lf_lazy = pl.scan_parquet(ruta_parquet)
t_lazy = time.time() - t0
print(f"⏱️ Tiempo de escaneo estructurado (Lazy): {t_lazy:.4f} segundos")

⏱️ Tiempo de carga Completa en RAM (Eager): 0.3928 segundos
⏱️ Tiempo de escaneo estructurado (Lazy): 0.0002 segundos


In [ ]:
# Analizar el tamaño del efecto real de las variantes significativas
resumen_efectos = df_eager.filter(pl.col("pval") < 0.000005).select([
    pl.col("effect_size").abs().max().alias("Efecto Máximo Absoluto"),
    pl.col("effect_size").abs().mean().alias("Efecto Promedio"),
    pl.col("maf").min().alias("MAF Mínimo")
])

print(resumen_efectos)

shape: (1, 3)
┌────────────────────────┬─────────────────┬────────────┐
│ Efecto Máximo Absoluto ┆ Efecto Promedio ┆ MAF Mínimo │
│ ---                    ┆ ---             ┆ ---        │
│ f64                    ┆ f64             ┆ f64        │
╞════════════════════════╪═════════════════╪════════════╡
│ 0.596892               ┆ 0.104816        ┆ null       │
└────────────────────────┴─────────────────┴────────────┘


## 2. 👁️ Celda 3: Verificación del "Predicate Pushdown" (Filtro en disco)

In [ ]:
# 1. Definimos el umbral de corte estadístico
UMBRAL_PVAL = 0.000005

# 2. Filtramos de forma Lazy las variantes que superan la prueba estadística
consulta_significativas = lf_lazy.filter(
    (pl.col("disorder") == "bpd") & 
    (pl.col("pval") < UMBRAL_PVAL)
).select([
    "variant_id", "chr", "pos", "pval", "effect_size"
]).sort("pval") # Ordenamos de menor a mayor (las más significativas primero)

# 3. Traemos los resultados a memoria
df_top_eficientes = consulta_significativas.collect()

# 4. Mostrar los resultados en pantalla
print(f"🧬 Se encontraron {len(df_top_eficientes):,} variantes altamente significativas.\n")

if len(df_top_eficientes) > 0:
    print("📝 NOTA METODOLÓGICA:")
    print(f"Las siguientes variantes presentan un pval < {UMBRAL_PVAL}. Esto demuestra una ")
    print("asociación estadística sólida con el Trastorno Bipolar, reduciendo la probabilidad ")
    print("de que este hallazgo sea un falso positivo debido al azar.\n")
    
    # Mostramos las top 10 más potentes
    print(df_top_eficientes.head(10))
else:
    print("⚠️ No se encontraron variantes por debajo del umbral especificado.")

🧬 Se encontraron 1,763 variantes altamente significativas.

📝 NOTA METODOLÓGICA:
Las siguientes variantes presentan un pval < 5e-06. Esto demuestra una 
asociación estadística sólida con el Trastorno Bipolar, reduciendo la probabilidad 
de que este hallazgo sea un falso positivo debido al azar.

shape: (10, 5)
┌─────────────────┬─────┬───────────┬────────────┬─────────────┐
│ variant_id      ┆ chr ┆ pos       ┆ pval       ┆ effect_size │
│ ---             ┆ --- ┆ ---       ┆ ---        ┆ ---         │
│ str             ┆ str ┆ i64       ┆ f64        ┆ f64         │
╞═════════════════╪═════╪═══════════╪════════════╪═════════════╡
│ 9:140251458:A:G ┆ 9   ┆ 140251458 ┆ 9.9530e-12 ┆ 0.161602    │
│ 9:140257189:C:T ┆ 9   ┆ 140257189 ┆ 1.3390e-10 ┆ -0.153501   │
│ 9:140258802:A:G ┆ 9   ┆ 140258802 ┆ 1.4840e-10 ┆ 0.153198    │
│ 9:140258412:A:C ┆ 9   ┆ 140258412 ┆ 1.6810e-10 ┆ 0.152697    │
│ 9:140259932:C:T ┆ 9   ┆ 140259932 ┆ 1.7560e-10 ┆ -0.152604   │
│ 9:140265782:C:T ┆ 9   ┆ 140265782 ┆ 

## 3. 🗂️ Celda 4: Análisis de Outliers Computado en Segundos

In [ ]:
# Probemos el algoritmo IQR en una columna numérica completa de forma Lazy
columna_prueba = "effect_size"

# Tomamos una muestra rápida para estimar los cuartiles
muestra_rapida = df_eager[columna_prueba].sample(n=50000, seed=42).drop_nulls()
Q1 = muestra_rapida.quantile(0.25)
Q3 = muestra_rapida.quantile(0.75)
IQR = Q3 - Q1
inf, sup = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

# Contamos los outliers en los millones de filas del archivo real
total_outliers = lf_lazy.filter(
    (pl.col(columna_prueba) < inf) | (pl.col(columna_prueba) > sup)
).select(pl.len()).collect().item()

print(f"📊 Columna analizada: {columna_prueba}")
print(f"Rango normal estimado: [{inf:.4f} a {sup:.4f}]")
print(f"Cantidad exacta de Outliers en TODO el archivo: {total_outliers:,} de {len(df_eager):,} filas total.")

📊 Columna analizada: effect_size
Rango normal estimado: [-0.1527 a 0.1524]
Cantidad exacta de Outliers en TODO el archivo: 180,761 de 6,201,203 filas total.
